In [ ]:
import os
import json
import math
import re
import sqlite3
import uuid
from collections import Counter
from datetime import datetime
from pathlib import Path
from typing import Annotated, List, Literal, Optional, TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    RemoveMessage,
    SystemMessage,
)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

# Cargar API Key gratuita de Google AI Studio
load_dotenv()
api_key = os.environ["GEMINI_API_KEY"].strip().strip('"').strip("'")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=api_key,
    temperature=0.1,
)

# Asegurar entorno de datos de negocio (reutilizando ops_company.db y SKILL.md)
os.makedirs("skills/ops-procurement", exist_ok=True)

skill_path = Path("skills/ops-procurement/SKILL.md")
if not skill_path.exists():
    skill_path.write_text(
        """---
name: ops-procurement
description: Procedimiento estándar para auditar desviaciones de ventas regionales y generar propuestas de compra de equipamiento.
---

# Reglas de Negocio y Aprobación
1. **Criterio de desviación:** Una región se considera en alerta crítica si sus ingresos están más de un 10% por debajo del objetivo trimestral.
2. **Límites presupuestarios de equipamiento:**
   - Hasta 3.000 EUR: Aprobación estándar de Responsable de Operaciones.
   - Más de 3.000 EUR: Requiere escalado obligatorio a Dirección Financiera (CFO) en el borrador.
3. **Stock:** Nunca proponer una compra que supere el stock disponible en catálogo.

# Plantilla Obligatoria del Informe Final
## 1. Diagnóstico Regional
- Región afectada, ingresos vs. objetivo y desviación porcentual.
## 2. Propuesta de Equipamiento y Cumplimiento
- Detalle de SKU, cantidad, coste total y nivel de aprobación requerido.
## 3. Borrador de Orden Generado
- ID del borrador y estado de validación.
""",
        encoding="utf-8",
    )

conn = sqlite3.connect("ops_company.db")
cur = conn.cursor()
cur.execute(
    """CREATE TABLE IF NOT EXISTS regional_sales (
        region TEXT, quarter TEXT, revenue_eur REAL, target_eur REAL, headcount INTEGER
    )"""
)
cur.execute("DELETE FROM regional_sales")
cur.executemany(
    "INSERT INTO regional_sales VALUES (?, ?, ?, ?, ?)",
    [
        ("Norte", "Q3", 420000, 400000, 18),
        ("Sur", "Q3", 265000, 350000, 14),
        ("Centro", "Q3", 510000, 500000, 22),
        ("Levante", "Q3", 310000, 320000, 15),
    ],
)
cur.execute(
    """CREATE TABLE IF NOT EXISTS hardware_catalog (
        sku TEXT PRIMARY KEY, item_name TEXT, unit_price_eur REAL, stock INTEGER
    )"""
)
cur.execute("DELETE FROM hardware_catalog")
cur.executemany(
    "INSERT INTO hardware_catalog VALUES (?, ?, ?, ?)",
    [
        ("LAP-PRO-16", "Portátil Pro 16GB RAM", 1250.0, 10),
        ("LAP-DEV-32", "Portátil Workstation 32GB RAM", 2100.0, 4),
        ("MON-27-4K", "Monitor 27 pulgadas 4K", 380.0, 15),
    ],
)
cur.execute(
    """CREATE TABLE IF NOT EXISTS purchase_drafts (
        draft_id TEXT PRIMARY KEY,
        thread_id TEXT,
        region TEXT,
        sku TEXT,
        quantity INTEGER,
        total_eur REAL,
        approval_level TEXT,
        status TEXT,
        created_at TEXT
    )"""
)
conn.commit()
conn.close()
print("Entorno LangGraph + Gemini + SQLite inicializado correctamente.")

In [ ]:
# Almacén de Memoria a Largo Plazo (Semántica + Episódica en SQLite)
class ExtractedMemory(BaseModel):
    memory_type: Literal["semantic", "episodic"] = Field(
        description="'semantic' para reglas, hechos o preferencias permanentes; 'episodic' para decisiones pasadas, compras o incidentes ocurridos."
    )
    content: str = Field(description="Descripción concisa y autocontenida del recuerdo.")
    importance: int = Field(ge=1, le=5, description="Importancia de 1 a 5.")

class ExtractedMemoryList(BaseModel):
    memories: List[ExtractedMemory]

class LongTermMemoryStore:
    """Almacena y recupera recuerdos semánticos y episódicos entre sesiones distintas."""

    def __init__(self, db_path: str = "long_term_memory.db"):
        self.db_path = db_path
        self._init_db()

    def _init_db(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                """CREATE TABLE IF NOT EXISTS memories (
                    id TEXT PRIMARY KEY,
                    user_id TEXT,
                    memory_type TEXT,
                    content TEXT,
                    importance INTEGER,
                    created_at TEXT
                )"""
            )

    @staticmethod
    def _tokenize(text: str) -> Counter:
        tokens = re.findall(r"\b[a-záéíóúñ0-9\-]{3,}\b", text.lower())
        return Counter(tokens)

    def _cosine_similarity(self, text_a: str, text_b: str) -> float:
        vec_a, vec_b = self._tokenize(text_a), self._tokenize(text_b)
        if not vec_a or not vec_b:
            return 0.0
        common = set(vec_a.keys()) & set(vec_b.keys())
        dot = sum(vec_a[k] * vec_b[k] for k in common)
        norm_a = math.sqrt(sum(v * v for v in vec_a.values()))
        norm_b = math.sqrt(sum(v * v for v in vec_b.values()))
        return dot / (norm_a * norm_b)

    def save_memory(self, user_id: str, memory_type: str, content: str, importance: int = 3):
        with sqlite3.connect(self.db_path) as conn:
            # Evitar duplicados exactos
            exists = conn.execute(
                "SELECT 1 FROM memories WHERE user_id = ? AND content = ?",
                (user_id, content),
            ).fetchone()
            if not exists:
                conn.execute(
                    "INSERT INTO memories VALUES (?, ?, ?, ?, ?, ?)",
                    (
                        f"MEM-{uuid.uuid4().hex[:6].upper()}",
                        user_id,
                        memory_type,
                        content,
                        importance,
                        datetime.now().isoformat(),
                    ),
                )

    def retrieve(self, user_id: str, query: str, top_k: int = 4) -> List[dict]:
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.execute(
                "SELECT * FROM memories WHERE user_id = ?", (user_id,)
            ).fetchall()

        scored = []
        for row in rows:
            sim = self._cosine_similarity(query, row["content"])
            # Combinamos similitud semántica léxica + peso de importancia
            score = sim + (row["importance"] * 0.05)
            scored.append((score, dict(row)))

        scored.sort(key=lambda x: x[0], reverse=True)
        return [item[1] for item in scored[:top_k]]

    def consolidate_from_interaction(self, user_id: str, interaction_summary: str):
        """Usa el LLM para destilar nuevos recuerdos semánticos y episódicos tras cerrar una tarea."""
        extractor = llm.with_structured_output(ExtractedMemoryList)
        prompt = (
            "Analiza la siguiente interacción operativa y extrae únicamente hechos duraderos "
            "(memoria 'semantic', ej. preferencias de hardware o límites) o eventos relevantes ocurridos "
            "(memoria 'episodic', ej. órdenes de compra aprobadas/rechazadas para una región).\n\n"
            f"Interacción:\n{interaction_summary}"
        )
        result: ExtractedMemoryList = extractor.invoke(prompt)
        for mem in result.memories:
            self.save_memory(user_id, mem.memory_type, mem.content, mem.importance)

# Inicializamos el almacén y cargamos recuerdos históricos previos de la empresa
ltm_store = LongTermMemoryStore()
ltm_store.save_memory(
    "director_ops",
    "semantic",
    "Preferencia de Dirección: Para refuerzos comerciales en regiones críticas usar siempre el modelo LAP-PRO-16, reservando LAP-DEV-32 exclusivamente para perfiles de ingeniería.",
    importance=5,
)
ltm_store.save_memory(
    "director_ops",
    "episodic",
    "Histórico Q2: Se aprobó la compra de 2 monitores MON-27-4K para Levante (DRAFT-Q2001); el proveedor tardó 7 días hábiles en entregar.",
    importance=3,
)
print("Almacén de Memoria a Largo Plazo listo con recuerdos semánticos y episódicos iniciales.")

In [ ]:
# Context Engineering (Compresión Rolling Summary + Window Trimming)
MAX_SHORT_TERM_MESSAGES = 4  # Mantenemos solo los últimos 4 mensajes crudos en ventana activa

def estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4)

def compress_medium_term_context(messages: List[BaseMessage], current_summary: str) -> tuple[str, List[RemoveMessage]]:
    """
    Evita que la ventana de contexto explote en conversaciones largas:
    1. Si hay más de MAX_SHORT_TERM_MESSAGES, resume los más antiguos en `rolling_summary`.
    2. Devuelve comandos `RemoveMessage` para purgar los mensajes antiguos del estado activo.
    """
    if len(messages) <= MAX_SHORT_TERM_MESSAGES:
        return current_summary, []

    messages_to_compress = messages[:-MAX_SHORT_TERM_MESSAGES]
    formatted_old = "\n".join(
        f"- {m.type.upper()}: {m.content}" for m in messages_to_compress
    )

    compression_prompt = (
        "Eres un ingeniero de contexto. Actualiza el resumen ejecutivo de la conversación "
        "incorporando los mensajes antiguos sin perder cifras, SKUs, regiones ni decisiones clave.\n\n"
        f"Resumen acumulado previo:\n{current_summary or 'Ninguno'}\n\n"
        f"Nuevos mensajes a compactar:\n{formatted_old}\n\n"
        "Devuelve un resumen denso y estructurado de máximo 120 palabras."
    )
    new_summary = llm.invoke(compression_prompt).content
    removals = [RemoveMessage(id=m.id) for m in messages_to_compress if m.id is not None]
    return new_summary, removals

print("Motor de Context Engineering (Rolling Summary + Trimming) configurado.")

In [ ]:
# Grafo Cíclico (StateGraph) con Autocorrección, Persistencia e HITL

# Definición del Estado Persistente del Grafo
class CopilotState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    user_id: str
    rolling_summary: str
    retrieved_memories: str
    business_metrics: str
    skill_rules: str
    draft_proposal: dict
    final_report: str
    audit_passed: bool
    audit_feedback: str
    retry_count: int
    order_status: str

# Esquemas Pydantic para salidas estructuradas
class DraftOutput(BaseModel):
    region: str
    sku: str
    quantity: int
    unit_price_eur: float
    total_eur: float
    approval_level: str
    markdown_report: str

class AuditVerdict(BaseModel):
    passed: bool = Field(description="True solo si cumple TODAS las reglas de SKILL.md, stock y memoria semántica.")
    issues: List[str] = Field(description="Lista de incumplimientos detectados (vacía si passed=True).")
    fix_instructions: str = Field(description="Instrucciones concretas para corregir el borrador en el siguiente ciclo.")

# Nodos del Grafo
def context_and_memory_node(state: CopilotState) -> dict:
    """Paso 1: Comprime contexto a medio plazo y recupera memoria semántica/episódica a largo plazo."""
    messages = state.get("messages", [])
    current_summary = state.get("rolling_summary", "")
    user_id = state.get("user_id", "director_ops")

    # Context Engineering: compactar mensajes si la ventana crece demasiado
    new_summary, removals = compress_medium_term_context(messages, current_summary)

    # Recuperar memoria a largo plazo relevante para la última petición
    latest_query = messages[-1].content if messages else ""
    memories = ltm_store.retrieve(user_id=user_id, query=latest_query, top_k=3)
    formatted_memories = "\n".join(
        f"- [{m['memory_type'].upper()}] {m['content']}" for m in memories
    ) or "Sin recuerdos previos relevantes."

    print(f"[Context & Memory] Mensajes activos: {len(messages) - len(removals)} | Recuerdos recuperados: {len(memories)}")
    return {
        "messages": removals,
        "rolling_summary": new_summary,
        "retrieved_memories": formatted_memories,
        "retry_count": 0,
        "audit_feedback": "",
    }

def data_analyst_node(state: CopilotState) -> dict:
    """Paso 2: Consulta SQLite de ventas y catálogo de hardware."""
    with sqlite3.connect("ops_company.db") as conn:
        conn.row_factory = sqlite3.Row
        sales = [dict(r) for r in conn.execute("SELECT * FROM regional_sales WHERE quarter = 'Q3'").fetchall()]
        catalog = [dict(r) for r in conn.execute("SELECT * FROM hardware_catalog").fetchall()]

    for s in sales:
        s["deviation_pct"] = round(((s["revenue_eur"] - s["target_eur"]) / s["target_eur"]) * 100, 2)

    skill_text = Path("skills/ops_procurement/SKILL.md").read_text(encoding="utf-8")
    metrics_json = json.dumps({"sales_q3": sales, "hardware_catalog": catalog}, ensure_ascii=False, indent=2)

    print("[Data Analyst] Métricas Q3 y catálogo cargados desde ops_company.db.")
    return {
        "business_metrics": metrics_json,
        "skill_rules": skill_text,
    }

def drafter_node(state: CopilotState) -> dict:
    """Paso 3 (Nodo cíclico): Genera o corrige la propuesta usando SKILL.md, memoria y feedback del Auditor."""
    attempt = state.get("retry_count", 0) + 1
    feedback = state.get("audit_feedback", "")
    latest_request = state["messages"][-1].content

    print(f"[Drafter] Generando borrador e informe (Iteración #{attempt})...")

    structured_drafter = llm.with_structured_output(DraftOutput)
    prompt = (
        "Eres el especialista de Operaciones y Compras. Genera una propuesta de compra y un informe ejecutivo "
        "cumpliendo estrictamente la Skill corporativa y los recuerdos a largo plazo.\n\n"
        f"PETICIÓN DEL USUARIO:\n{latest_request}\n\n"
        f"RESUMEN CONVERSACIÓN PREVIA (Medio Plazo):\n{state.get('rolling_summary', 'N/A')}\n\n"
        f"MEMORIA A LARGO PLAZO (Semántica y Episódica):\n{state['retrieved_memories']}\n\n"
        f"DATOS REALES DE NEGOCIO (SQLite):\n{state['business_metrics']}\n\n"
        f"SKILL CORPORATIVA OBLIGATORIA:\n{state['skill_rules']}\n\n"
    )
    if feedback:
        prompt += (
            f"ATENCIÓN — TU BORRADOR ANTERIOR FUE RECHAZADO POR EL AUDITOR.\n"
            f"CORRIGE OBLIGATORIAMENTE ESTOS PUNTOS:\n{feedback}\n"
        )

    draft: DraftOutput = structured_drafter.invoke(prompt)
    draft_dict = draft.model_dump()
    draft_dict["draft_id"] = f"DRAFT-{uuid.uuid4().hex[:6].upper()}"

    return {
        "draft_proposal": draft_dict,
        "final_report": draft.markdown_report,
        "retry_count": attempt,
    }

def auditor_node(state: CopilotState) -> dict:
    """Paso 4 (Nodo Critic): Audita el borrador contra SKILL.md, stock y memoria semántica."""
    draft = state["draft_proposal"]
    structured_auditor = llm.with_structured_output(AuditVerdict)

    audit_prompt = (
        "Eres el Auditor Interno de Cumplimiento. Revisa el borrador propuesto frente a las reglas "
        "de negocio, el stock real en base de datos y las preferencias de memoria semántica.\n"
        "Sé estricto: si la cantidad supera el stock disponible, si una compra > 3.000 EUR no indica "
        "escalado a Dirección Financiera (CFO), si se elige un modelo prohibido por la memoria semántica "
        "o si faltan las 3 secciones Markdown de la Skill, marca passed=False.\n\n"
        f"BORRADOR A AUDITAR:\n{json.dumps(draft, ensure_ascii=False, indent=2)}\n\n"
        f"DATOS DE STOCK Y VENTAS:\n{state['business_metrics']}\n\n"
        f"MEMORIA SEMÁNTICA:\n{state['retrieved_memories']}\n\n"
        f"REGLAS SKILL.md:\n{state['skill_rules']}"
    )
    verdict: AuditVerdict = structured_auditor.invoke(audit_prompt)

    if verdict.passed:
        print(f"[Auditor] APROBADO en el intento #{state['retry_count']}.")
        return {"audit_passed": True, "audit_feedback": ""}
    else:
        print(f"[Auditor] RECHAZADO en el intento #{state['retry_count']}. Motivos: {verdict.issues}")
        return {
            "audit_passed": False,
            "audit_feedback": f"Problemas detectados: {verdict.issues}. Instrucciones: {verdict.fix_instructions}",
        }

def route_after_audit(state: CopilotState) -> Literal["human_approval", "drafter"]:
    """Arista condicional cíclica: reintenta hasta 3 veces si el Auditor rechaza el borrador."""
    if state["audit_passed"]:
        return "human_approval"
    if state["retry_count"] < 3:
        print(f"   --> [Ciclo] Devolviendo control al Drafter para autocorrección (reintento {state['retry_count']}/3)...")
        return "drafter"
    print("   --> [Escalado] Máximo de reintentos alcanzado; escalando con advertencia al humano.")
    return "human_approval"

def human_approval_node(state: CopilotState) -> dict:
    """Paso 5: Pausa durable (HITL) mediante interrupt(). Guarda la orden en SQLite si se aprueba."""
    draft = state["draft_proposal"]
    # interrupt() detiene el grafo y guarda todo el estado en SqliteSaver
    human_decision = interrupt(
        {
            "mensaje": "Revisión humana requerida antes de registrar la orden en el ERP.",
            "draft_id": draft["draft_id"],
            "region": draft["region"],
            "sku": draft["sku"],
            "quantity": draft["quantity"],
            "total_eur": draft["total_eur"],
            "approval_level": draft["approval_level"],
            "audit_passed": state["audit_passed"],
        }
    )

    if str(human_decision).strip().lower() == "s":
        with sqlite3.connect("ops_company.db") as conn:
            conn.execute(
                "INSERT INTO purchase_drafts VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                (
                    draft["draft_id"],
                    state.get("user_id", "director_ops"),
                    draft["region"],
                    draft["sku"],
                    draft["quantity"],
                    draft["total_eur"],
                    draft["approval_level"],
                    "APPROVED_AND_PERSISTED",
                    datetime.now().isoformat(),
                ),
            )
        # Consolidamos memoria episódica a largo plazo para futuras sesiones
        ltm_store.consolidate_from_interaction(
            state.get("user_id", "director_ops"),
            f"Orden aprobada {draft['draft_id']}: {draft['quantity']}x {draft['sku']} por {draft['total_eur']} EUR para región {draft['region']} (aprobación: {draft['approval_level']}).",
        )
        status = "APPROVED_AND_PERSISTED"
        reply = f"Orden {draft['draft_id']} aprobada, guardada en ops_company.db y consolidada en memoria episódica.\n\n{state['final_report']}"
    else:
        status = "REJECTED_BY_HUMAN"
        reply = f"Orden {draft['draft_id']} cancelada por el usuario."

    return {
        "order_status": status,
        "messages": [AIMessage(content=reply)],
    }

# Construcción y compilación del Grafo Cíclico con Checkpointer Persistente
builder = StateGraph(CopilotState)
builder.add_node("context_and_memory", context_and_memory_node)
builder.add_node("data_analyst", data_analyst_node)
builder.add_node("drafter", drafter_node)
builder.add_node("auditor", auditor_node)
builder.add_node("human_approval", human_approval_node)

builder.add_edge(START, "context_and_memory")
builder.add_edge("context_and_memory", "data_analyst")
builder.add_edge("data_analyst", "drafter")
builder.add_edge("drafter", "auditor")
builder.add_conditional_edges(
    "auditor",
    route_after_audit,
    {"drafter": "drafter", "human_approval": "human_approval"},
)
builder.add_edge("human_approval", END)

# Conexión SQLite persistente entre reinicios de Kernel
checkpoint_conn = sqlite3.connect("langgraph_checkpoints.db", check_same_thread=False)
checkpointer = SqliteSaver(checkpoint_conn)

copilot_graph = builder.compile(checkpointer=checkpointer)
print("Grafo cíclico compilado con SqliteSaver persistente.")

In [ ]:
# Probamos una petición que obliga al sistema a consultar la memoria semántica
# (que prohíbe LAP-DEV-32 para equipos comerciales) o ajustar reglas de negocio.
config_sesion_1 = {"configurable": {"thread_id": "ops-thread-001"}}

result = copilot_graph.invoke(
    {
        "user_id": "director_ops",
        "messages": [
            HumanMessage(
                content="Analiza Q3, detecta la región en alerta crítica y prepara compra de 3 portátiles workstation LAP-DEV-32 para reforzar a sus comerciales."
            )
        ],
    },
    config=config_sesion_1,
)

print("\n--- ESTADO PAUSADO EN INTERRUPT (HITL) ---")
print(result.get("__interrupt__"))

In [ ]:
# Ejecuta esta celda después de revisar el interrupt (puedes incluso reiniciar el Kernel
# y ejecutar solo Celda 1, 2, 3, 4 y esta Celda 6 para comprobar la persistencia en disco).
config_sesion_1 = {"configurable": {"thread_id": "ops-thread-001"}}

# Inspeccionamos el historial de checkpoints guardados (Time-Travel)
history = list(copilot_graph.get_state_history(config_sesion_1))
print(f"Checkpoints guardados en SQLite para este hilo: {len(history)}")

# Reanudamos la ejecución inyectando la aprobación humana 's'
final_state = copilot_graph.invoke(Command(resume="s"), config=config_sesion_1)

print("\n=== INFORME FINAL TRAS APROBACIÓN ===")
print(final_state["messages"][-1].content)

In [ ]:
# Abrimos un hilo COMPLETAMENTE NUEVO ('ops-thread-002') y comprobamos si recuerda
# la decisión episódica que acabamos de aprobar en 'ops-thread-001'
memories_for_new_session = ltm_store.retrieve(
    user_id="director_ops",
    query="¿Qué órdenes de compra aprobamos recientemente para la región Sur?",
    top_k=3,
)
print("=== MEMORIA A LARGO PLAZO RECUPERADA EN UNA NUEVA SESIÓN ===")
for m in memories_for_new_session:
    print(f"- [{m['memory_type'].upper()}] {m['content']}")

# Simulamos una conversación larga de 8 mensajes para observar la compresión de medio plazo
long_history = [
    HumanMessage(content=f"Consulta operativa #{i}: revisando métricas parciales de la región Sur.")
    for i in range(1, 8)
]
summary, removals = compress_medium_term_context(long_history, "")
print(f"\n=== CONTEXT ENGINEERING (MEDIO PLAZO) ===")
print(f"Mensajes purgados de la ventana activa: {len(removals)} de {len(long_history)}")
print(f"Rolling Summary generado ({estimate_tokens(summary)} tokens aprox.):\n{summary}")